# Lezione 5 — LangChain: dal modello all'applicazione

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ccasadei-maggioli/corso-nlp-genai-2026/blob/main/lezione_5_langchain/notebook_05_langchain.ipynb)

Bentornato! 👋 Nella Lezione 4 abbiamo usato un **LLM generativo** (Qwen2.5) scrivendo
i prompt "a mano" e incollando i dati dentro le stringhe. Funziona, ma in
un'applicazione vera servono pezzi **riutilizzabili** e **componibili**. È esattamente
ciò che fa **LangChain**: il framework per orchestrare gli LLM in applicazioni.

In questa lezione:
1. capiamo *cos'è* **LangChain** e perché ci semplifica la vita;
2. impariamo **LCEL** — il modo di collegare i componenti con l'operatore `|`;
3. **avvolgiamo** (wrap) il nostro Qwen2.5 dentro LangChain;
4. costruiamo **prompt template**, **catene**, **output parser** (anche JSON) e
   **memoria** conversazionale — un mattone alla volta, sempre sulle **recensioni**.

> 🎯 **Filo conduttore:** sempre le stesse **recensioni clienti in italiano**.
> Oggi impariamo a *strutturare* il lavoro con l'LLM: è il ponte verso l'applicazione
> finale con **RAG** (Lezione 6).

---
### ⚙️ Reminder: attiva la GPU T4
Menu **`Runtime` → `Change runtime type` → Hardware accelerator: `T4 GPU` → `Save`**.
Anche oggi carichiamo un modello 7B in 4-bit: **senza GPU il modello non si carica**.

> ⏳ **AVVISO sui tempi:** ogni `invoke` della catena fa girare l'LLM, quindi può
> richiedere **qualche secondo** (di più per i testi lunghi). È normale. Più avanti
> trovi l'alternativa più leggera **Qwen2.5-3B** se la T4 risulta lenta.

## 1. Installiamo le librerie

Oltre alle solite (`transformers`, `accelerate`, `bitsandbytes` per il modello 7B in
4-bit), oggi installiamo l'ecosistema **LangChain**:
- **`langchain`** — il framework e l'operatore `|` di **LCEL**;
- **`langchain-huggingface`** — l'adattatore per usare i modelli Hugging Face dentro
  LangChain (`HuggingFacePipeline`, `ChatHuggingFace`);
- **`langchain-community`** — componenti aggiuntivi della community.

> 💡 Come sempre, ogni notebook installa da sé ciò che gli serve, così puoi aprire
> questa lezione in modo indipendente.

In [ ]:
# -q = silenzioso. La prima installazione richiede un paio di minuti.
!pip install -q "transformers>=4.45" "accelerate>=0.34" "bitsandbytes>=0.44" "langchain>=0.3" "langchain-huggingface>=0.1" "langchain-community>=0.3"
print("Librerie installate ✅")

In [ ]:
import torch

print("Versione PyTorch:", torch.__version__)
print("GPU disponibile:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Scheda:", torch.cuda.get_device_name(0))
else:
    print("⚠️  GPU non attiva. Vai su Runtime > Change runtime type > T4 GPU,")
    print("    altrimenti il modello 7B NON si caricherà.")

## 2. Che cos'è LangChain? 🦜🔗

Nella Lezione 4 abbiamo scritto codice come: *"costruisci la lista di messaggi, applica
il chat template, tokenizza, chiama `generate`, decodifica solo la risposta"*. Ogni
volta. **LangChain** è il **framework** che incapsula questi passi in **componenti**
standard e riutilizzabili, e ci dà un modo pulito per **collegarli** tra loro.

I **mattoni** che vedremo oggi:

- **LLM / Chat model** — il modello (il nostro Qwen2.5), avvolto in un'interfaccia comune.
- **PromptTemplate** — un *modello* di prompt con dei "buchi" (`{variabile}`) da riempire:
  così lo scrivi una volta e lo riusi su mille recensioni.
- **OutputParser** — trasforma il testo grezzo del modello in qualcosa di **utilizzabile
  dal software**: una stringa pulita, una lista, un **JSON**, un oggetto.
- **Memoria** — fa "ricordare" alla catena i turni precedenti di una conversazione.
- **Retriever** *(anticipazione L6)* — recupera i documenti pertinenti a una domanda: è il
  pezzo che, unito a LLM + prompt, costruisce il **RAG**.

### LCEL — LangChain Expression Language
Il modo moderno di comporre i componenti è **LCEL**: si collegano con l'operatore **`|`**
(la "pipe"), proprio come nelle pipe della shell. L'output di un componente diventa
l'input del successivo:

```
catena = prompt | modello | parser
risultato = catena.invoke({"testo": "..."})
```

Si legge: *prendi le variabili, costruisci il prompt, passalo al modello, poi al parser.*
Ogni catena espone metodi comodi come `.invoke(...)` (una chiamata), `.batch([...])`
(molte chiamate) e `.stream(...)` (risposta token per token).

## 3. Il nostro dataset: recensioni clienti 🛒

Ricarichiamo le solite recensioni sintetiche in italiano (schema:
`id, data, prodotto, categoria, rating, titolo, testo`). Sono *riproducibili* (seed fisso)
e non richiedono download esterni.

In [ ]:
import os, torch
import pandas as pd

# Scarica lo script generatore se non è già nella sessione Colab.
if not os.path.exists("genera_recensioni.py"):
    !wget -q https://raw.githubusercontent.com/ccasadei-maggioli/corso-nlp-genai-2026/main/dati/genera_recensioni.py

import genera_recensioni

df = pd.DataFrame(genera_recensioni.genera_recensioni(n=200, seed=42))
print("Numero di recensioni:", len(df))
df.head(3)

## 4. Carichiamo Qwen2.5 e lo avvolgiamo in LangChain 🚀

Carichiamo **`Qwen/Qwen2.5-7B-Instruct`** in **4-bit** (come nella Lezione 4: è il
"trucco" per farlo stare nella T4). La novità di oggi è l'ultimo pezzo: **avvolgiamo** il
modello in due classi di LangChain.

- **`HuggingFacePipeline`** prende la `pipeline` di Hugging Face e la espone come un LLM di
  LangChain.
- **`ChatHuggingFace`** ci aggiunge sopra la gestione dei **ruoli** (system/user/assistant):
  applica **automaticamente il chat template di Qwen**, così non dobbiamo più farlo a mano
  come in L4. È questo l'oggetto che useremo nelle catene.

> ⏳ La **prima** esecuzione **scarica** il modello (qualche GB): **1–3 minuti**. Le volte
> successive nella stessa sessione è immediato. Buon momento per una pausa...

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
import torch

model_id = "Qwen/Qwen2.5-7B-Instruct"

# Quantizzazione 4-bit: comprime i pesi da 16 a 4 bit (~14GB -> ~5-6GB), così il 7B
# entra nella T4. I calcoli restano in fp16 per mantenere la qualità.
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)

# Tokenizer (testo <-> token, contiene anche il chat template) e modello sulla GPU.
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map="auto")

# pipeline di generazione di Hugging Face: è il "motore" che LangChain avvolgerà.
# return_full_text=False -> ci restituisce SOLO la risposta nuova, non anche il prompt.
gen = pipeline("text-generation", model=model, tokenizer=tokenizer,
               max_new_tokens=512, do_sample=True, temperature=0.7, top_p=0.9, return_full_text=False)

print("Pipeline pronta ✅  Memoria GPU usata: "
      f"{torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

# 1) Avvolge la pipeline di Hugging Face come "LLM" di LangChain.
llm = HuggingFacePipeline(pipeline=gen)

# 2) Aggiunge la gestione dei ruoli (system/user/assistant): applica automaticamente
#    il chat template di Qwen. 'chat' è l'oggetto che collegheremo nelle catene LCEL.
chat = ChatHuggingFace(llm=llm)

print("LLM avvolto in LangChain ✅")

> 💡 **Alternativa più leggera/veloce.** Se la T4 ti sembra lenta (o vai in *out of
> memory*), passa a **`Qwen/Qwen2.5-3B-Instruct`**: basta cambiare la riga
> `model_id = "Qwen/Qwen2.5-3B-Instruct"` qui sopra e rieseguire le celle. Risponde **più
> in fretta**, con qualità un po' inferiore ma adeguata per gli esempi di questa lezione.

## 5. Il primo prompt template + la prima catena LCEL 🔗

Invece di scrivere il prompt come una stringa fissa, usiamo un **`ChatPromptTemplate`**:
una struttura con un messaggio **`system`** (le istruzioni di fondo) e un messaggio
**`human`** con un "buco" `{testo}` che riempiremo con la recensione di turno.

Poi costruiamo la **catena LCEL** collegando tre componenti con la pipe `|`:

```
catena = prompt | chat | StrOutputParser()
```

- `prompt` riceve un dizionario di variabili e produce i messaggi;
- `chat` (il nostro Qwen) genera la risposta;
- `StrOutputParser()` estrae la **stringa** pulita dalla risposta del modello.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "Sei un analista di recensioni. Rispondi in italiano, in modo conciso."),
    ("human", "Riassumi in una frase questa recensione:\n{testo}")])

# La catena LCEL: prompt -> modello -> parser (estrae la stringa).
catena = prompt | chat | StrOutputParser()

# invoke riceve un dizionario con le variabili del template ({testo}).
catena.invoke({"testo": df.iloc[0]['testo']})

Il vantaggio del template è che lo **riusiamo** su qualsiasi recensione cambiando solo le
variabili. E poiché la catena espone `.batch(...)`, possiamo riassumere **molte**
recensioni in una sola chiamata.

In [ ]:
# Riassumiamo le prime 3 recensioni in un colpo solo con .batch(...).
risultati = catena.batch([{"testo": t} for t in df["testo"].head(3)])
for i, r in enumerate(risultati):
    print(f"[{i}] {r}\n")

## 6. Output strutturato in JSON 🧩

Una stringa va bene per gli umani, ma per **integrare l'LLM nel software** (salvare su
database, alimentare una dashboard, prendere decisioni nel codice) ci serve un output
**strutturato**. LangChain ci permette di chiederlo direttamente in **JSON**.

L'idea:
1. descriviamo la struttura desiderata con un modello **Pydantic** (`Analisi`);
2. **`JsonOutputParser`** genera, da quel modello, le **istruzioni di formato** da
   incollare nel prompt (`get_format_instructions()`) e poi fa il **parsing** della
   risposta in un dizionario Python;
3. `.partial(...)` "pre-riempie" la variabile `{istruzioni}` del template, una volta sola.

Il risultato di `catena_json.invoke(...)` non è più testo, ma un **dizionario** pronto da
usare nel codice (`risultato["sentiment"]`, `risultato["temi"]`, …).

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

# Descriviamo la struttura che vogliamo ottenere. I 'description' aiutano il modello.
class Analisi(BaseModel):
    sentiment: str = Field(description="uno tra: positivo, negativo, neutro")
    temi: list[str] = Field(description="temi citati, es. spedizione, prezzo, qualità")
    riassunto: str = Field(description="riassunto in una frase")

parser = JsonOutputParser(pydantic_object=Analisi)

# Il parser sa generare le istruzioni di formato (lo schema JSON da rispettare):
# le pre-iniettiamo nel system prompt con .partial(...).
prompt_json = ChatPromptTemplate.from_messages([
    ("system", "Estrai le informazioni dalla recensione e rispondi SOLO con JSON valido.\n{istruzioni}"),
    ("human", "{testo}")]).partial(istruzioni=parser.get_format_instructions())

# La catena: prompt -> modello -> parser JSON (restituisce un dizionario Python).
catena_json = prompt_json | chat | parser

catena_json.invoke({"testo": df.iloc[0]['testo']})

> ⚠️ **Nota onesta.** Con modelli relativamente piccoli il JSON **a volte** non è perfetto
> (una virgola di troppo, un campo mancante) e il parser può sollevare un errore. In
> **produzione** si gestiscono questi casi: si intercetta l'eccezione, si **riprova**, si
> usano parser più robusti (es. `OutputFixingParser`) o si abbassa la `temperature`. Qui
> ci basta capire il meccanismo.

In [ ]:
# Esempio di parsing robusto: se il JSON non è valido, non blocchiamo tutto.
def analizza_sicuro(testo):
    try:
        return catena_json.invoke({"testo": testo})
    except Exception as e:
        return {"errore": f"JSON non valido ({type(e).__name__})", "testo": testo[:60] + "..."}

for t in df["testo"].head(3):
    print(analizza_sicuro(t))

## 7. Memoria conversazionale 💬

Finora ogni `invoke` era indipendente: il modello **non ricorda** nulla del turno
precedente. Per costruire un **assistente conversazionale** (es. un chatbot che aiuta ad
analizzare le recensioni) serve la **memoria**.

In LCEL si ottiene con **`RunnableWithMessageHistory`**, che avvolge la catena e, a ogni
chiamata, inietta automaticamente lo **storico** dei messaggi nel punto indicato dal
**`MessagesPlaceholder("history")`** del prompt.

- `get_history(session_id)` restituisce (creandolo se serve) lo storico di **quella**
  conversazione — così sessioni diverse non si mescolano;
- `input_messages_key` e `history_messages_key` dicono al wrapper *quale* variabile è
  l'input dell'utente e *dove* mettere lo storico;
- il `session_id` nella `config` identifica la conversazione.

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

prompt_chat = ChatPromptTemplate.from_messages([
    ("system", "Sei un assistente che aiuta ad analizzare recensioni di prodotti."),
    MessagesPlaceholder("history"),   # qui verrà inserito lo storico della conversazione
    ("human", "{input}")])

catena_chat = prompt_chat | chat | StrOutputParser()

# Memoria in RAM: un dizionario session_id -> storico dei messaggi.
store = {}
def get_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Avvolge la catena aggiungendo la gestione automatica dello storico.
chat_con_memoria = RunnableWithMessageHistory(catena_chat, get_history,
        input_messages_key="input", history_messages_key="history")

print("Chat con memoria pronta ✅")

In [ ]:
# La config identifica la conversazione tramite session_id.
cfg = {"configurable": {"session_id": "demo"}}

# Primo turno: ci presentiamo.
print("Turno 1:")
print(chat_con_memoria.invoke({"input": "Ciao, di cosa ti occupi?"}, config=cfg))

In [ ]:
# Secondo turno (stessa session_id): il modello RICORDA il contesto del turno 1.
print("Turno 2:")
print(chat_con_memoria.invoke({"input": "Riassumi il tuo ruolo in 3 parole."}, config=cfg))

Cambiando `session_id` si ottiene una conversazione **nuova e separata** (storico vuoto):
è così che un'app servirebbe più utenti contemporaneamente senza mescolare i contesti.

## 8. Esercizio 🏋️

Tocca a te. **Estendi lo schema JSON** della sezione 6 aggiungendo un campo
**`valutazione_stimata`**: un intero da **1 a 5** che rappresenta le "stelle" che il
modello assegnerebbe alla recensione. Poi riadatta la catena JSON e provala.

Completa i punti **TODO**, quindi esegui.

> 💡 *Variante alternativa:* invece di modificare lo schema, costruisci una catena che,
> data una recensione, generi una **risposta dell'assistenza clienti** (system con tono
> cortese + `human` che passa il `{testo}` e chiede una risposta di 3-4 frasi).

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

# TODO: aggiungi il campo 'valutazione_stimata' (int, da 1 a 5) al modello.
class AnalisiEstesa(BaseModel):
    sentiment: str = Field(description="uno tra: positivo, negativo, neutro")
    temi: list[str] = Field(description="temi citati, es. spedizione, prezzo, qualità")
    riassunto: str = Field(description="riassunto in una frase")
    # TODO: scrivi qui il nuovo campo, es:
    # valutazione_stimata: int = Field(description="stelle stimate, intero da 1 a 5")

# TODO: crea il parser sul nuovo modello AnalisiEstesa.
parser_es = ...

# TODO: costruisci il prompt (come in sezione 6) iniettando le istruzioni del parser
#       con .partial(istruzioni=parser_es.get_format_instructions()).
prompt_es = ...

# TODO: componi la catena: prompt_es | chat | parser_es, poi invoca su df.iloc[0]['testo'].
# catena_es = ...
# catena_es.invoke({"testo": df.iloc[0]['testo']})

### ✅ Soluzione

Una possibile soluzione. L'unica differenza rispetto alla sezione 6 è il **campo in più**
nello schema Pydantic: il resto della catena è identico, e questo mostra quanto è comodo
**modificare l'output strutturato** senza toccare la logica.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

class AnalisiEstesa(BaseModel):
    sentiment: str = Field(description="uno tra: positivo, negativo, neutro")
    temi: list[str] = Field(description="temi citati, es. spedizione, prezzo, qualità")
    riassunto: str = Field(description="riassunto in una frase")
    valutazione_stimata: int = Field(description="stelle stimate per la recensione, intero da 1 a 5")

parser_es = JsonOutputParser(pydantic_object=AnalisiEstesa)

prompt_es = ChatPromptTemplate.from_messages([
    ("system", "Estrai le informazioni dalla recensione e rispondi SOLO con JSON valido.\n{istruzioni}"),
    ("human", "{testo}")]).partial(istruzioni=parser_es.get_format_instructions())

catena_es = prompt_es | chat | parser_es

risultato = catena_es.invoke({"testo": df.iloc[0]['testo']})
print(risultato)
# Confronto con le stelle reali presenti nel dataset:
print("\nStelle reali nel dataset:", df.iloc[0]['rating'])

## 9. Riepilogo e prossimi passi ✅

Oggi siamo passati dal *modello* all'*applicazione* con **LangChain**:
- capito il ruolo di LangChain e di **LCEL** (i componenti collegati con `|`);
- **avvolto** Qwen2.5 con `HuggingFacePipeline` + `ChatHuggingFace` (chat template
  automatico);
- costruito **prompt template** riutilizzabili e la prima **catena** `prompt | chat | parser`;
- ottenuto **output strutturato in JSON** con `JsonOutputParser` + Pydantic — il pezzo
  che rende l'LLM integrabile nel software;
- aggiunto la **memoria** conversazionale con `RunnableWithMessageHistory`.

Abbiamo anche ricordato l'**alternativa leggera** `Qwen/Qwen2.5-3B-Instruct` per la T4.

➡️ **Prossima lezione (Progetto finale — RAG):** mettiamo insieme tutti i mattoni del
corso. Useremo gli **embeddings** (L2) per cercare le recensioni pertinenti a una domanda
(il **retriever** anticipato oggi), e li daremo in pasto all'**LLM** tramite una **catena
LangChain**: è il **RAG** (Retrieval-Augmented Generation), il cuore dell'applicazione
finale che **risponde a domande sulle recensioni citando le fonti**.

📦 Tutto il materiale del corso: https://github.com/ccasadei-maggioli/corso-nlp-genai-2026